# Week 2 — Notebook 2: Technical Indicators

Implement **10 technical indicators** from scratch using only `pandas` and `numpy`.
No `ta`, `ta-lib`, or similar libraries — every value must come from the OHLCV columns.

These features are what the LSTM model will consume in later weeks.
The final cell saves a complete feature CSV to `data/processed/`.

> Implement every TODO using the mathematical formula given.

## 0. Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR      = Path('.')
RAW_DIR       = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def safe_symbol(s: str) -> str:
    return s.replace('^','').replace('.','_').replace('=','_').replace('-','_')

In [3]:
import yfinance as yf

RAW_DIR.mkdir(parents=True, exist_ok=True)

for symbol in ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', '^NSEI',
               'AAPL', 'MSFT', 'NVDA', 'TSLA', 'SPY',
               'BTC-USD', 'ETH-USD', 'BNB-USD',
               'EURUSD=X', 'GBPUSD=X', 'USDINR=X']:
    try:
        data = yf.download(symbol, start='2015-01-01', auto_adjust=False, progress=False)
        data.columns = [col[0] for col in data.columns]
        data = data.reset_index()
        data.columns = [str(c).lower().replace(' ', '_') for c in data.columns]
        data['symbol'] = symbol
        data.to_csv(RAW_DIR / f'{safe_symbol(symbol)}.csv', index=False)
        print(f'Saved {symbol}')
    except Exception as e:
        print(f'Failed {symbol}: {e}')

print('All done!')

Saved RELIANCE.NS
Saved TCS.NS
Saved INFY.NS
Saved HDFCBANK.NS
Saved ^NSEI
Saved AAPL
Saved MSFT
Saved NVDA
Saved TSLA
Saved SPY
Saved BTC-USD
Saved ETH-USD
Saved BNB-USD
Saved EURUSD=X
Saved GBPUSD=X
Saved USDINR=X
All done!


In [5]:
# TODO 0.1 — Load a symbol from RAW_DIR
# Use 'SPY' or any symbol you downloaded in Notebook 1.
SYMBOL='SPY'
# - parse 'date' as datetime index, sort ascending
df=pd.read_csv(RAW_DIR/f'{safe_symbol(SYMBOL)}.csv',parse_dates=['date'])
df=df.sort_values('date').set_index('date')
# - ensure columns: open, high, low, close, volume (all lowercase)
# - replace 0-volume rows with NaN
df['volume']=df['volume'].replace(0,np.nan)


In [6]:
# Run this after TODO 0.1 is complete

close  = df['close']
high   = df['high']
low    = df['low']
volume = df['volume']

In [12]:
# TODO 0.2 — Compute base return features needed for the model
# These are not indicators, but inputs the LSTM will use directly.
#
# return_1
# return_5
# log_return_1

df['return_1']     = close.pct_change(1)  #percentage change from yesterday to today
df['return_5']     = close.pct_change(5)  #percentage change from 5days ago to today
df['log_return_1'] = np.log(close/close.shift(1)) #today closing / yesterday closing and taking log of that
df[['return_1','return_5','log_return_1']].head()

,return_1,return_5,log_return_1
date,,,
2015-01-02,NaN,NaN,NaN
2015-01-05,-0.018060,NaN,-0.018225
2015-01-06,-0.009419,NaN,-0.009464
2015-01-07,0.012461,NaN,0.012384
2015-01-08,0.017745,NaN,0.017589


## 1. Simple Moving Average (SMA)

Smooths price noise and reveals the prevailing trend direction. Price above SMA → uptrend; below → downtrend.

$$\text{SMA}_n(t) = \frac{1}{n}\sum_{i=0}^{n-1}C_{t-i}$$

We also need the **ratio** of close to SMA, which tells the model how far price has stretched from its average:

$$\text{sma\_ratio}_n = \frac{C_t}{\text{SMA}_n(t)} - 1$$

In [17]:
# TODO 1.1 — Compute SMA_20, SMA_50, SMA_200
#             and sma_ratio_20, sma_ratio_50

df['sma_20']      = close.rolling(20).mean() #rolling(20)-look at last 20 days and .mean() -cal avg
df['sma_50']      = close.rolling(50).mean()
df['sma_200']     = close.rolling(200).mean()
df['sma_ratio_20'] = close/df['sma_20']-1
df['sma_ratio_50'] = close/df['sma_50']-1
df[['sma_20','sma_50','sma_200','sma_ratio_20','sma_ratio_50']].head()

,sma_20,sma_50,sma_200,sma_ratio_20,sma_ratio_50
date,,,,,
2015-01-02,NaN,NaN,NaN,NaN,NaN
2015-01-05,NaN,NaN,NaN,NaN,NaN
2015-01-06,NaN,NaN,NaN,NaN,NaN
2015-01-07,NaN,NaN,NaN,NaN,NaN
2015-01-08,NaN,NaN,NaN,NaN,NaN


In [19]:
# TODO 1.2 — Plot close + SMA_20 + SMA_50 (last 2 years only)
# Mark Golden Cross (SMA_50 crosses above SMA_200) and Death Cross as vertical lines
# Hint: a cross occurs when the sign of (sma_50 - sma_200) changes

 recent = df.last('2Y') #takes only last 2 years data
 cross  = np.sign(recent['sma_50']-recent['sma_200']).diff() #find where crosses happens,+ve if sma 50 above sma 200 and -ve where sma 200 above sma 50
 golden = cross[cross>0].index #get dates where cross is +ve
 death  = cross[cross<0]

# Plot Graph
 plt.figure(figsize=(14,5))
 plt.plot(recent.index,recent['close'],label='Close',linewidth=1.2)
 plt.plot(recent.index,recent['sma_20'],label='SMA 20',linewidth=1)
 plt.plot(recent.index,recent['sma_50'],label='SMA 50',linewidth=1)
 plt.plot(recent.index,recent['sma_200'],label='SMA 200',linewidth=1)
 for d in golden:
   plt.axvline(d,color='gold',alpha=0.7,linestyle='--',label='Golden Cross')
 for d in death:
   plt.axvline(d,color='black',alpha=0.7,linestyle='--',label='Death Cross')
 plt.title(f'{SYMBOL}-Close & SMAs(last 2 years)')
 plt.xlabel('Date')
 plt.ylabel('Price')
 plt.legend(fontsize=8)
 plt.grid(True)
 plt.tight_layout()
 plt.show()

IndentationError: unexpected indent (343925494.py, line 5)

## 2. Exponential Moving Average (EMA)

Assigns exponentially decaying weights to past prices — more reactive to recent moves than SMA.

$$k = \frac{2}{n+1}, \qquad \text{EMA}(t) = C_t \cdot k + \text{EMA}(t-1)\cdot(1-k)$$

Also compute:
$$\text{ema\_ratio}_{20} = \frac{C_t}{\text{EMA}_{20}(t)} - 1$$

In [ ]:
# TODO 2.1 — Compute EMA_12, EMA_20, EMA_26 and ema_ratio_20

# df['ema_12']       = ...
# df['ema_20']       = ...
# df['ema_26']       = ...
# df['ema_ratio_20'] = ...

In [ ]:
# TODO 2.2 — On a single chart, plot Close + SMA_20 + EMA_20 for the last 365 days
# Visually confirm: EMA reacts faster to price turns than SMA


## 3. Relative Strength Index (RSI)

Momentum oscillator bounded in $[0, 100]$. Measures the ratio of average gains to average losses over $n$ days.

$$\Delta_t = C_t - C_{t-1}$$
$$G_t = \max(\Delta_t,\,0), \quad L_t = |\min(\Delta_t,\,0)|$$
$$\overline{G}_n = \text{EMA}_n(G), \quad \overline{L}_n = \text{EMA}_n(L)$$
$$RS = \frac{\overline{G}_n}{\overline{L}_n}, \quad \text{RSI} = 100 - \frac{100}{1 + RS}$$

Wilder's RSI uses $\alpha = 1/n$ (i.e. `com = n-1`) rather than the standard EMA span. RSI $> 70$: overbought. RSI $< 30$: oversold. Normalise to $[0, 1]$ before storing as a model feature.

In [ ]:
# TODO 3.1 — Implement RSI with window=14; store normalised (÷100) as 'rsi_14'
# Use Wilder smoothing: ewm(com=n-1, adjust=False)

# delta    = ...
# gain     = ...
# loss     = ...
# avg_gain = ...
# avg_loss = ...
# rs       = ...
# df['rsi_14'] = ...

In [ ]:
# TODO 3.2 — Two-panel chart: close (top) + RSI (bottom), sharex=True
# Dashed lines at 0.70 and 0.30; shade overbought in light red, oversold in light green


## 4. MACD

Captures the convergence/divergence of two EMAs. Three outputs:

$$\text{MACD Line} = \text{EMA}_{12} - \text{EMA}_{26}$$
$$\text{Signal Line} = \text{EMA}_9(\text{MACD Line})$$
$$\text{Histogram} = \text{MACD Line} - \text{Signal Line}$$

Histogram sign-change → momentum shift. MACD crossing above Signal → bullish.

In [ ]:
# TODO 4.1 — Compute macd, macd_signal, macd_hist using ema_12 and ema_26

# df['macd']        = ...
# df['macd_signal'] = ...
# df['macd_hist']   = ...

In [ ]:
# TODO 4.2 — Three-panel chart: close | macd+signal | histogram coloured by sign


## 5. Bollinger Bands

Places volatility-scaled envelopes around the 20-day SMA.

$$\text{Upper} = \text{SMA}_{20} + k\cdot\sigma_{20}, \quad \text{Lower} = \text{SMA}_{20} - k\cdot\sigma_{20}, \quad k=2$$

Two derived features used as model inputs:

| Feature | Formula |
|---|---|
| **BB Width** | $\frac{\text{Upper} - \text{Lower}}{C_t}$ — proxy for volatility |
| **%B** | $\frac{C_t - \text{Lower}}{\text{Upper} - \text{Lower}}$ — position within bands |

In [ ]:
# TODO 5 — Compute bb_upper, bb_lower, bb_width, bb_pct_b

# rolling_std    = ...
# df['bb_upper'] = ...
# df['bb_lower'] = ...
# df['bb_width'] = ...
# df['bb_pct_b'] = ...   # clip to [0, 1] optional but useful

## 6. Rate of Change (ROC)

Measures momentum as the percentage change over $n$ periods.

$$\text{ROC}_n(t) = \frac{C_t - C_{t-n}}{C_{t-n}} \times 100$$

Compute for $n = 5$ and $n = 10$.

In [ ]:
# TODO 6 — Compute roc_5 and roc_10

# df['roc_5']  = ...
# df['roc_10'] = ...

## 7. Average True Range (ATR)

Measures volatility using all three price extremes. Unlike rolling return std, ATR accounts for overnight gaps.

$$TR_t = \max\bigl(H_t - L_t,\;|H_t - C_{t-1}|,\;|L_t - C_{t-1}|\bigr)$$

$$\text{ATR}_{14}(t) = \text{EMA}_{14}(TR) \quad\text{(Wilder smoothing: }\alpha = 1/n\text{)}$$

Normalise by close to make it comparable across symbols:
$$\text{atr\_ratio} = \frac{\text{ATR}_{14}}{C_t}$$

In [ ]:
# TODO 7 — Compute true_range, atr_14, atr_ratio
# Use Wilder smoothing (ewm with com=13, adjust=False)

# prev_close = ...
# true_range = ...
# df['atr_14']    = ...
# df['atr_ratio'] = ...

## 8. Stochastic Oscillator (%K and %D)

Compares the closing price to the price range over a lookback window.

$$\%K_n(t) = \frac{C_t - \min_n(L)}{\max_n(H) - \min_n(L)} \times 100$$

$$\%D = \text{SMA}_3(\%K) \quad \text{(signal smoothing)}$$

Use $n = 14$. Normalise both to $[0, 1]$. Store as `stoch_k` and `stoch_d`.

In [ ]:
# TODO 8.1 — Compute stoch_k and stoch_d

# lowest_low   = ...
# highest_high = ...
# df['stoch_k'] = ...
# df['stoch_d'] = ...

In [ ]:
# TODO 8.2 — Plot stoch_k and stoch_d, last 365 days
# Mark 0.80 and 0.20 thresholds
# When %K crosses above %D from below the oversold zone → classic buy signal


## 9. On-Balance Volume (OBV)

Cumulative volume indicator — adds volume on up-days, subtracts it on down-days. Used to detect whether volume confirms or diverges from a price trend.

$$\text{OBV}(t) = \text{OBV}(t-1) + \begin{cases} +V_t & C_t > C_{t-1} \\ -V_t & C_t < C_{t-1} \\ 0 & C_t = C_{t-1} \end{cases}$$

Store the **percentage change** of OBV as the model feature: $\text{obv\_change} = \frac{\text{OBV}(t) - \text{OBV}(t-1)}{|\text{OBV}(t-1)|}$

In [ ]:
# TODO 9 — Compute obv and obv_change
# np.sign(close.diff()) gives +1 / 0 / -1 for each day's direction

# direction       = ...
# df['obv']       = ...
# df['obv_change']= ...

## 10. Volume-Weighted Average Price (VWAP)

The price level weighted by volume — a fair-value benchmark widely used by institutional traders.

We compute a **rolling 20-day VWAP** using the typical price:

$$P_{\text{typ}}(t) = \frac{H_t + L_t + C_t}{3}$$

$$\text{VWAP}_{20}(t) = \frac{\sum_{i=0}^{19} P_{\text{typ}}(t{-}i)\cdot V_{t{-}i}}{\sum_{i=0}^{19} V_{t{-}i}}$$

Normalise: $\text{vwap\_ratio} = C_t / \text{VWAP} - 1$

In [ ]:
# TODO 10 — Compute rolling VWAP (20-day) and vwap_ratio

# typical_price   = ...
# tp_vol          = ...
# df['vwap']       = ...
# df['vwap_ratio'] = ...

## 11. Build & Save the Feature Dataset

Combine all indicators into a clean DataFrame and save to `data/processed/`.
This CSV will be loaded directly by the model notebooks in later weeks.

In [ ]:
FEATURE_COLUMNS = [
    'return_1', 'return_5', 'log_return_1',
    'sma_ratio_20', 'sma_ratio_50',
    'ema_ratio_20',
    'rsi_14',
    'macd', 'macd_signal',
    'roc_5', 'roc_10',
    'bb_width', 'bb_pct_b',
    'atr_ratio',
    'stoch_k', 'stoch_d',
    'obv_change',
    'vwap_ratio',
]

In [ ]:
# TODO 11.1 — Add target columns, clean, and save
# 'future_return'    = next-day return  (close shifted -1 divided by close, minus 1)
# 'target_direction' = 1 if future_return > 0, else 0
#
# Then:
#   1. Replace inf / -inf with NaN
#   2. Drop rows where ANY required column is NaN
#   3. Reset index so 'date' becomes a regular column again
#   4. Save to  PROCESSED_DIR / f'{safe_symbol(SYMBOL)}_features.csv'
#   5. Print the output path and shape; show .describe() of required columns

# df['future_return']    = ...
# df['target_direction'] = ...

# required = ...
# df       = ...
# df_clean = ...

# out_path = ...
# df_clean.to_csv(...)
# print(...)
# df_clean[required].describe().round(4)

In [ ]:
# TODO 11.2 (Challenge) — Run the full pipeline for ALL downloaded symbols
# Loop over every CSV in RAW_DIR, compute all indicators, save to PROCESSED_DIR.
# Wrap each symbol in try/except so one failure does not stop the loop.
# Print a summary table at the end with columns:
#   symbol | rows_raw | rows_processed | drop_pct

# summary = []
# for csv_path in ...:
#     sym = ...
#     try:
#         raw       = ...
#         # ... compute all indicators on raw ...
#         processed = ...
#         # ... save to PROCESSED_DIR ...
#         summary.append({...})
#     except Exception as e:
#         summary.append({'symbol': sym, 'error': str(e)})

# pd.DataFrame(summary)

---
